In [14]:
import pandas as pd
import torch

# Per riproducibilità
torch.manual_seed(8347247)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

df = pd.read_csv("../data/all_ships.csv")

In [15]:
from sklearn.model_selection import train_test_split

X = df[["sex", "age", "age_missing", "class", "crew"]]
Y = df["survived"]

X_tensor = torch.tensor(X.values, dtype=torch.float32)
Y_tensor = torch.tensor(Y.values, dtype=torch.long) # La CrossEntropyLoss richiede target long

# per riproducibilità si usa random_state fissato
X_train, X_test, Y_train, Y_test = train_test_split(X_tensor, Y_tensor, test_size=0.2, random_state=42, stratify=Y_tensor)

mean = X_train.mean(0)
std = X_train.std(0)

X_train_norm = (X_train - mean) / std
X_test_norm = (X_test - mean) / std

In [16]:
from torch.utils.data import TensorDataset, DataLoader

train_ds = TensorDataset(X_train_norm, Y_train)
test_ds = TensorDataset(X_test_norm, Y_test)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=64)

In [17]:
from torch import nn

class MLP_2Layer(nn.Module):
    """
    Semplice MLP con:
        - input_dim = 5 (feature)
        - hidden_dim = 3 (layer nascosto)
        - output_dim = 2 (classi: morto / sopravvissuto)
    Architettura:
        input -> Linear(5,3) -> ReLU -> Linear(3,2) -> logits
    """
    def __init__(self, in_features, hidden_dim, out_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, hidden_dim), # rappresentazione nascosta (batch_size, 3)
            nn.ReLU(), # funzione di attivazione
            nn.Linear(hidden_dim, out_features) # punteggi non normalizzati (batch_size, 2)
        )

    def forward(self, x):
        return self.net(x) 

In [18]:
class DeepMLP(nn.Module):
    """
    Modello MLP profondo definito dinamicamente.
    hidden_units: lista es. [8, 4] crea due hidden layer 5->8->4->2
    """
    def __init__(self, in_features, hidden_units, out_features):
        super().__init__()
        
        # costruiamo i layer uno dopo l'altro, aggiungendo ReLU tra di essi
        layers = []
        input_dim = in_features
        for hidden_dim in hidden_units:
            layers.append(nn.Linear(input_dim, hidden_dim))
            layers.append(nn.ReLU())
            input_dim = hidden_dim

        # layer di output (logits)
        layers.append(nn.Linear(input_dim, out_features))

        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

In [19]:
from torch.optim import SGD
from torch.utils.tensorboard import SummaryWriter
from sklearn.metrics import accuracy_score

def train_model(model, train_loader, test_loader, lr=0.05, epochs=300):
    writer = SummaryWriter(f'../results/{model._get_name()}')
    model = model.to(device)

    weights = torch.tensor([1.0, 2.0]).to(device)
    criterion = nn.CrossEntropyLoss(weight=weights)
    optimizer = SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=0.001)

    for epoch in range(epochs):
        model.train()

        train_loss = 0.0
        y_true = []
        y_pred = []
        
        for X_batch, Y_batch in train_loader:
            X_batch = X_batch.to(device)
            Y_batch = Y_batch.to(device)

            output = model(X_batch)
            loss = criterion(output, Y_batch)

            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            train_loss += loss.item() * X_batch.size(0)

            # l'output è array di logits quindi si prende il punteggio più alto che indica la classe più probabile
            preds = output.argmax(dim=1) 
            y_true.extend(Y_batch.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

        train_loss /= len(train_loader.dataset)
        train_acc = accuracy_score(y_true, y_pred)

        model.eval()

        test_loss = 0.0
        y_true = []
        y_pred = []

        with torch.no_grad():
            for X_batch, Y_batch in test_loader:
                X_batch = X_batch.to(device)
                Y_batch = Y_batch.to(device)

                output = model(X_batch)

                loss = criterion(output, Y_batch)
                test_loss += loss.item() * X_batch.size(0)
                
                preds = output.argmax(dim=1)
                y_true.extend(Y_batch.cpu().numpy())
                y_pred.extend(preds.cpu().numpy())

        test_loss /= len(test_loader.dataset)
        test_acc = accuracy_score(y_true, y_pred)

        writer.add_scalar('loss/train', train_loss, epoch)
        writer.add_scalar('accuracy/train', train_acc, epoch)
        writer.add_scalar('loss/test', test_loss, epoch)
        writer.add_scalar('accuracy/test', test_acc, epoch)

        if epoch % 50 == 0:
            print(f"Epoch {epoch+1}/{epochs} | train_loss {train_loss:.4f} | train_acc {train_acc:.4f} | test_loss {test_loss:.4f} | test_acc {test_acc:.4f}")

    print(f"Epoch {epoch+1}/{epochs} | train_loss {train_loss:.4f} | train_acc {train_acc:.4f} | test_loss {test_loss:.4f} | test_acc {test_acc:.4f}\n")
    writer.close()

    return model, train_loss, train_acc, test_loss, test_acc

In [20]:
from sklearn.metrics import precision_score, recall_score, f1_score

def evaluate_model(model):
    model.eval()

    with torch.no_grad():
        output = model(X_test_norm.to(device))
        probs = output.argmax(dim=1)

    y_true = Y_test.cpu().numpy()
    y_pred = probs.cpu().numpy()

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    return accuracy, precision, recall, f1

In [ ]:
# Addestramento modello base
base_model = MLP_2Layer(in_features=5, hidden_dim=3, out_features=2)
base_model, base_train_loss, base_train_acc, base_test_loss, base_test_acc = train_model(base_model, train_loader, test_loader)
base_accuracy, base_precision, base_recall, base_f1 = evaluate_model(base_model)

# Addestramento modello deep
deep_model = DeepMLP(in_features=5, hidden_units=[15, 7, 4], out_features=2)
deep_model, deep_train_loss, deep_train_acc, deep_test_loss, deep_test_acc = train_model(deep_model, train_loader, test_loader)
deep_accuracy, deep_precision, deep_recall, deep_f1 = evaluate_model(deep_model)

Epoch 1/300 | train_loss 0.6584 | train_acc 0.6341 | test_loss 0.6374 | test_acc 0.6307
Epoch 51/300 | train_loss 0.6139 | train_acc 0.6637 | test_loss 0.6096 | test_acc 0.6554
Epoch 101/300 | train_loss 0.6159 | train_acc 0.6667 | test_loss 0.6036 | test_acc 0.6398
Epoch 151/300 | train_loss 0.6129 | train_acc 0.6709 | test_loss 0.6125 | test_acc 0.6372
Epoch 201/300 | train_loss 0.6125 | train_acc 0.6605 | test_loss 0.6315 | test_acc 0.6320
Epoch 251/300 | train_loss 0.6176 | train_acc 0.6514 | test_loss 0.6105 | test_acc 0.6723
Epoch 300/300 | train_loss 0.6240 | train_acc 0.6475 | test_loss 0.6285 | test_acc 0.6073

Epoch 1/300 | train_loss 0.6948 | train_acc 0.5609 | test_loss 0.6933 | test_acc 0.6749
Epoch 51/300 | train_loss 0.5992 | train_acc 0.6341 | test_loss 0.5914 | test_acc 0.6229
Epoch 101/300 | train_loss 0.6040 | train_acc 0.6390 | test_loss 0.5998 | test_acc 0.5904
Epoch 151/300 | train_loss 0.6052 | train_acc 0.6266 | test_loss 0.6108 | test_acc 0.6398
Epoch 201/300 |

In [22]:
print("Model Comparison\n")

print("MLP_2Layer")
print(f"train_loss: {base_train_loss:.4f}")
print(f"train_acc: {base_train_acc:.4f}")
print(f"test_loss: {base_test_loss:.4f}")
print(f"test_acc: {base_test_acc:.4f}\n")

print(f"Accuracy: {base_accuracy:.4f}")
print(f"Precision: {base_precision:.4f}")
print(f"Recall: {base_recall:.4f}")
print(f"F1 Score: {base_f1:.4f}\n")

print("DeepMLP")
print(f"train_loss: {deep_train_loss:.4f}")
print(f"train_acc: {deep_train_acc:.4f}")
print(f"test_loss: {deep_test_loss:.4f}")
print(f"test_acc: {deep_test_acc:.4f}\n")

print(f"Accuracy: {deep_accuracy:.4f}")
print(f"Precision: {deep_precision:.4f}")
print(f"Recall: {deep_recall:.4f}")
print(f"F1 Score: {deep_f1:.4f}")

Model Comparison

MLP_2Layer
train_loss: 0.6240
train_acc: 0.6475
test_loss: 0.6285
test_acc: 0.6073

Accuracy: 0.6073
Precision: 0.4372
Recall: 0.7240
F1 Score: 0.5452

DeepMLP
train_loss: 0.6014
train_acc: 0.6348
test_loss: 0.5923
test_acc: 0.6229

Accuracy: 0.6229
Precision: 0.4535
Recall: 0.7800
F1 Score: 0.5735
